# Capítulo 09 · Errores y excepciones

*Traducción y adaptación de [A Whirlwind Tour of Python](https://github.com/jakevdp/WhirlwindTourOfPython)
de Jake VanderPlas (O'Reilly, 2016), publicado con licencia [CC0](https://creativecommons.org/publicdomain/zero/1.0/deed.es).
Adaptación: [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es).*

📖 Lectura complementaria del curso de Python de MiAtlas.cl · se abre en Colab o en JupyterLite, sin GPU.

No importa cuán hábil seas programando: tarde o temprano cometerás un error.
Esos errores son de tres tipos básicos:

- *Errores de sintaxis:* el código no es Python válido (en general, fáciles de arreglar).
- *Errores en tiempo de ejecución:* código sintácticamente válido que falla al ejecutarse, quizás por una entrada no
  válida (a veces fáciles de arreglar).
- *Errores semánticos:* errores de lógica; el código se ejecuta sin problemas, pero el resultado no es el que
  esperabas (a menudo muy difíciles de encontrar y arreglar).

Aquí nos concentraremos en cómo tratar de forma limpia los *errores en tiempo de ejecución*.
Como veremos, Python los maneja con su sistema de *manejo de excepciones*.

> 📝 **Nota de MiAtlas.cl:** varias celdas de este cuaderno fallan **a propósito** para mostrar la excepción. Están
> marcadas con la etiqueta `raises-exception`, así que puedes ejecutar todo el cuaderno de una vez y seguirá hasta el
> final.

## Errores en tiempo de ejecución

Si has programado algo en Python, seguro te has topado con errores en tiempo de ejecución.
Pueden aparecer de muchas formas.

Por ejemplo, si intentas usar una variable que no está definida:

In [ ]:
print(Q)

O si intentas una operación que no está definida:

In [ ]:
1 + 'abc'

O quizás intentas calcular un resultado que no está definido matemáticamente:

In [ ]:
2 / 0

O quizás intentas acceder a un elemento de una secuencia que no existe:

In [ ]:
L = [1, 2, 3]
L[1000]

Fíjate en que, en cada caso, Python no se limita a indicar que hubo un error: entrega una excepción *con significado*,
que dice qué salió mal exactamente y en qué línea de código ocurrió.
Tener errores con significado como estos es muy útil cuando tratas de encontrar el origen de los problemas en tu
código.

> 📝 **Nota de MiAtlas.cl:** los mensajes de error mejoraron mucho desde 2016. Desde Python 3.10 y 3.11, el intérprete
> marca con `^^^^` la parte exacta de la línea que falló y, si escribes mal un nombre, sugiere el correcto
> (`NameError: name 'nombr' is not defined. Did you mean: 'nombre'?`). Los mensajes del intérprete están en inglés.

## Capturar excepciones: `try` y `except`

La herramienta principal que te da Python para manejar excepciones en tiempo de ejecución es la cláusula
`try`...`except`. Su estructura básica es esta:

In [ ]:
try:
    print("esto se ejecuta primero")
except:
    print("esto se ejecuta solo si hay un error")

Fíjate en que el segundo bloque no se ejecutó: eso es porque el primer bloque no produjo ningún error.
Pongamos una instrucción problemática en el bloque `try` y veamos qué pasa:

In [ ]:
try:
    print("intentemos algo:")
    x = 1 / 0  # ZeroDivisionError
except:
    print("¡algo salió mal!")

Aquí vemos que, cuando se lanzó el error dentro del `try` (en este caso, un `ZeroDivisionError`), el error se capturó
y se ejecutó el bloque `except`.

Un uso frecuente de esto es revisar la entrada de datos dentro de una función u otra parte del código.
Por ejemplo, podríamos querer una función que capture la división por cero y devuelva otro valor, quizás un número
convenientemente grande como $10^{100}$:

In [ ]:
def division_segura(a, b):
    try:
        return a / b
    except:
        return 1E100

In [ ]:
division_segura(1, 2)

In [ ]:
division_segura(2, 0)

Sin embargo, este código tiene un problema sutil: ¿qué pasa cuando aparece otro tipo de excepción? Por ejemplo, esto
probablemente no es lo que queríamos:

In [ ]:
division_segura(1, '2')

Dividir un entero por una cadena lanza un `TypeError`, que nuestro código, demasiado entusiasta, capturó y trató como
si fuera un `ZeroDivisionError`.
Por eso, casi siempre es mejor capturar las excepciones de forma *explícita*:

In [ ]:
def division_segura(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return 1E100

In [ ]:
division_segura(1, 0)

In [ ]:
division_segura(1, '2')

Ahora capturamos solo los errores de división por cero y dejamos pasar sin cambios todos los demás.

> 📝 **Nota de MiAtlas.cl:** un `except:` sin tipo captura **todo**, incluso la interrupción con `Ctrl-C`
> (`KeyboardInterrupt`) y la salida del programa (`SystemExit`). Si de verdad necesitas capturar cualquier error,
> escribe `except Exception:`, que deja pasar esas dos. Y si puedes, nombra el tipo concreto, como arriba.

## Lanzar excepciones: `raise`

Hemos visto lo valioso que es tener excepciones informativas al usar partes del lenguaje Python.
Es igual de valioso usar excepciones informativas en el código que escribes, para que quienes lo usen (¡empezando por
ti!) puedan entender qué causó sus errores.

Para lanzar tus propias excepciones se usa la instrucción `raise`. Por ejemplo:

In [ ]:
raise RuntimeError("mi mensaje de error")

Como ejemplo de dónde puede ser útil, volvamos a la función `fibonacci` que definimos antes:

In [ ]:
def fibonacci(N):
    L = []
    a, b = 0, 1
    while len(L) < N:
        a, b = b, a + b
        L.append(a)
    return L

Un posible problema es que el valor de entrada podría ser negativo.
Hoy eso no provoca ningún error en nuestra función, pero quizás queramos avisar que un `N` negativo no está permitido.
Por convención, los errores que vienen de valores de parámetros no válidos lanzan un `ValueError`:

In [ ]:
def fibonacci(N):
    if N < 0:
        raise ValueError("N no puede ser negativo")
    L = []
    a, b = 0, 1
    while len(L) < N:
        a, b = b, a + b
        L.append(a)
    return L

In [ ]:
fibonacci(10)

In [ ]:
fibonacci(-10)

Ahora quien use la función sabe exactamente por qué la entrada no es válida, ¡e incluso podría usar un bloque
`try`...`except` para manejarlo!

In [ ]:
N = -10
try:
    print("intentando esto...")
    print(fibonacci(N))
except ValueError:
    print("Valor incorrecto: hay que hacer otra cosa")

## Profundizar en las excepciones

Quiero mencionar brevemente otros conceptos con los que te podrías encontrar.
No entraré en detalle sobre cómo y por qué usarlos; solo te muestro la sintaxis para que los explores por tu cuenta.

### Acceder al mensaje de error

A veces, en una instrucción `try`...`except`, quieres trabajar con el propio mensaje de error.
Eso se hace con la palabra clave `as`:

In [ ]:
try:
    x = 1 / 0
except ZeroDivisionError as err:
    print("La clase del error es:  ", type(err))
    print("El mensaje del error es:", err)

Con este patrón puedes personalizar aún más el manejo de excepciones de tu función.

### Definir excepciones propias

Además de las excepciones incorporadas, puedes definir excepciones propias mediante la *herencia de clases*.
Por ejemplo, si quieres un tipo especial de `ValueError`, puedes hacer esto:

In [ ]:
class MiErrorEspecial(ValueError):
    pass

raise MiErrorEspecial("este es el mensaje")

Esto te permite usar un bloque `try`...`except` que capture solo este tipo de error:

In [ ]:
try:
    print("haz algo")
    raise MiErrorEspecial("[aquí va un mensaje de error informativo]")
except MiErrorEspecial:
    print("haz otra cosa")

Esto te puede servir a medida que escribas código más personalizado.

## `try`...`except`...`else`...`finally`

Además de `try` y `except`, puedes usar las palabras clave `else` y `finally` para ajustar aún más el manejo de
excepciones. La estructura básica es esta:

In [ ]:
try:
    print("intenta algo aquí")
except:
    print("esto ocurre solo si falla")
else:
    print("esto ocurre solo si funciona")
finally:
    print("esto ocurre pase lo que pase")

La utilidad de `else` aquí es clara, pero ¿para qué sirve `finally`?
La cláusula `finally` se ejecuta realmente *pase lo que pase*: se suele usar para hacer algún tipo de limpieza después
de que termina una operación.

> 📝 **Nota de MiAtlas.cl:** dos novedades posteriores a 2016 que verás en código actual:
>
> - `raise NuevoError(...) from err` encadena una excepción con la que la causó, y el mensaje muestra ambas.
> - Desde Python 3.11 existen los *grupos de excepciones* (`ExceptionGroup`) y la cláusula `except*`, pensados para
>   código que puede fallar de varias formas a la vez, como las tareas concurrentes. Al empezar no los necesitarás.

## Pruébalo tú

Escribe una función que reciba un texto y devuelva el entero que representa, usando `int(texto)`.
Si el texto no es un número, `int` lanza un `ValueError`: captúralo (solo ese tipo) y devuelve `0`.
Guarda la función (sin llamarla) en `resultado`.

In [ ]:
resultado = None

# Escribe aquí tu función y luego: resultado = nombre_de_tu_funcion

In [ ]:
# Comprobación
if resultado is None:
    print("Aún no lo resuelves.")
elif callable(resultado) and resultado("42") == 42 and resultado("hola") == 0 and resultado("-7") == -7:
    print("✅ ¡Correcto!")
else:
    print("❌ Revisa tu respuesta")

**Sigue en MiAtlas.cl:** tema «Excepciones».

**Original:** [09-Errors-and-Exceptions.ipynb](https://github.com/jakevdp/WhirlwindTourOfPython/blob/master/09-Errors-and-Exceptions.ipynb)